# Proyecto QTranslate LIFIA — Quirk → OpenQASM 3.0

1. Prueba los algoritmos cuánticos del repositorio de origen.
2. Traduce circuitos de [Quirk](https://algassert.com/quirk) a OpenQASM 3.0.


In [8]:
import sys
import subprocess
import importlib.util

print('Python:', sys.executable)

packages = {
    'qiskit': 'qiskit',
    'matplotlib': 'matplotlib',
    'qiskit_aer': 'qiskit-aer',
    'pylatexenc': 'pylatexenc',
    'qiskit_qasm3_import': 'qiskit-qasm3-import',
}

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

print('Todas las dependencias necesarias están disponibles para este kernel.')


Python: c:\Users\PC\desarrollo\QTransLIFIA\.venv\Scripts\python.exe
Todas las dependencias necesarias están disponibles para este kernel.


In [19]:
from utils import qutils


def quirk_col_to_qasm3(col, offset):
    lines = []

    if 'Swap' in col:
        swap_indices = [k for k, g in enumerate(col) if g == 'Swap']
        if len(swap_indices) == 2:
            lines.append(f'swap q[{swap_indices[0] + offset}], q[{swap_indices[1] + offset}];')
        return lines

    if '•' in col:
        control_indices = [k for k, g in enumerate(col) if g == '•']
        target_gate = None
        for gate in ('X', 'Z', 'Y', 'X^½', 'X^-½', 'X^¼', 'X^-¼', 'Y^½', 'Y^-½', 'Y^¼', 'Y^-¼', 'Z^½', 'Z^-½', 'Z^¼', 'Z^-¼'):
            if gate in col:
                target_gate = gate
                target_index = col.index(gate)
                break
        if target_gate is None:
            return lines

        n_controls = len(control_indices)
        ctrl_str = ', '.join(f'q[{i + offset}]' for i in control_indices)
        tgt_str = f'q[{target_index + offset}]'

        ctrl_rz = {'Z^½': 'pi/2', 'Z^-½': '-pi/2', 'Z^¼': 'pi/4', 'Z^-¼': '-pi/4'}
        ctrl_rx = {'X^½': 'pi/2', 'X^-½': '-pi/2', 'X^¼': 'pi/4', 'X^-¼': '-pi/4'}
        ctrl_ry = {'Y^½': 'pi/2', 'Y^-½': '-pi/2', 'Y^¼': 'pi/4', 'Y^-¼': '-pi/4'}
        ctrl_prefix = ' '.join(['ctrl @'] * n_controls)

        if target_gate == 'X':
            lines.append(f'{ctrl_prefix} x {ctrl_str}, {tgt_str};')
        elif target_gate == 'Z':
            lines.append(f'{ctrl_prefix} z {ctrl_str}, {tgt_str};')
        elif target_gate == 'Y':
            lines.append(f'{ctrl_prefix} y {ctrl_str}, {tgt_str};')
        elif target_gate in ctrl_rz:
            lines.append(f'{ctrl_prefix} p({ctrl_rz[target_gate]}) {ctrl_str}, {tgt_str};')
        elif target_gate in ctrl_rx:
            lines.append(f'{ctrl_prefix} rx({ctrl_rx[target_gate]}) {ctrl_str}, {tgt_str};')
        elif target_gate in ctrl_ry:
            lines.append(f'{ctrl_prefix} ry({ctrl_ry[target_gate]}) {ctrl_str}, {tgt_str};')
        return lines

    for i, gate in enumerate(col):
        if gate in (1, '1', None):
            continue
        qi = i + offset
        mapping = {
            'Measure': f'c[{qi}] = measure q[{qi}];',
            'H': f'h q[{qi}];',
            'X': f'x q[{qi}];',
            'Y': f'y q[{qi}];',
            'Z': f'z q[{qi}];',
            'X^½': f'rx(pi/2) q[{qi}];',
            'X^-½': f'rx(-pi/2) q[{qi}];',
            'X^¼': f'rx(pi/4) q[{qi}];',
            'X^-¼': f'rx(-pi/4) q[{qi}];',
            'Y^½': f'ry(pi/2) q[{qi}];',
            'Y^-½': f'ry(-pi/2) q[{qi}];',
            'Y^¼': f'ry(pi/4) q[{qi}];',
            'Y^-¼': f'ry(-pi/4) q[{qi}];',
            'Z^½': f's q[{qi}];',
            'Z^-½': f'sdg q[{qi}];',
            'Z^¼': f't q[{qi}];',
            'Z^-¼': f'tdg q[{qi}];',
        }
        if gate in mapping:
            lines.append(mapping[gate])
    return lines


def quirk_to_qasm_qasm3(url, offset=0):
    circuito = qutils.parse_quirk_url(url)
    cols = circuito.get('cols', [])
    n = max((len(c) for c in cols), default=0) + offset
    lines = ['OPENQASM 3.0;', 'include "stdgates.inc";', f'qubit[{n}] q;', f'bit[{n}] c;', '']
    for col in cols:
        lines.extend(quirk_col_to_qasm3(col, offset))
    return '\n'.join(lines)


qutils.quirk_to_qasm = quirk_to_qasm_qasm3
quirk_to_qasm = quirk_to_qasm_qasm3


In [20]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.qutils import parse_quirk_url, quirk_col_to_qasm, quirk_to_qasm, quirk_circuit_info, encode_quirk_url


In [21]:
import json
from pathlib import Path
from IPython.display import display, Markdown


def load_algorithms():
    candidates = [
        Path.cwd() / "algorithms.json",
        Path.cwd() / "notebooks" / "algorithms.json",
        Path.cwd().parent / "notebooks" / "algorithms.json",
    ]
    for path in candidates:
        if path.exists():
            with path.open("r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError("No se encontró el archivo algorithms.json")


ALGORITHMS = load_algorithms()

def describe_algorithms():
    md = '| # | Algoritmo | Qubits | Descripción |\n|---|---|---|---|\n'
    for k, v in ALGORITHMS.items():
        info = quirk_circuit_info(v['url'])
        nq = info['n_qubits']
        md += f'| {k.split(".")[0]} | **{k.split(". ")[1]}** | {nq} | {v["desc"]} |\n'
    display(Markdown(md))

describe_algorithms()


| # | Algoritmo | Qubits | Descripción |
|---|---|---|---|
| 1 | **Shor** | 4 | Algoritmo de factorización de Shor (4 qubits) |
| 2 | **Bernstein-Vazirani** | 4 | Algoritmo de Bernstein-Vazirani (4 qubits) |
| 3 | **Grover** | 2 | Algoritmo de búsqueda de Grover (2 qubits) |
| 4 | **Deutsch-Jozsa** | 4 | Algoritmo de Deutsch-Jozsa (4 qubits) |
| 5 | **Simon** | 6 | Algoritmo de Simon (6 qubits) |
| 6 | **TSP** | 3 | Circuito para problema del viajante (TSP, 3 qubits) |
| 7 | **Teleportation** | 5 | Protocolo de teleportación cuántica (5 qubits) |
| 8 | **Phase Estimation** | 4 | Estimación de fase cuántica (4 qubits + compuertas custom) |
| 9 | **QFT** | 3 | Transformada de Fourier Cuántica (3 qubits) |
| 10 | **QAOA** | 2 | Quantum Approximate Optimization Algorithm (2 qubits) |
| 11 | **Kickback** | 2 | Phase kickback (2 qubits) |
| 12 | **Full Adder** | 4 | Sumador completo (4 qubits) |
| 13 | **Multicontroled Gates** | 5 | Circuito de prueba con compuertas multicontroladas CCX, CCY, CCZ, CCCX, CCCY y CCCZ |


In [22]:
import os
import pandas as pd
from pathlib import Path


output_dir = Path.cwd() / "algorithms_qasm"
output_dir.mkdir(exist_ok=True)


def generate_qasm_table_and_files():
    rows = []
    for name, data in ALGORITHMS.items():
        url = data["url"]
        info = quirk_circuit_info(url)
        qasm_code = quirk_to_qasm(url, data.get("offset", 0))

        safe_name = name.replace(".", "_").replace(" ", "_")
        file_path = output_dir / f"{safe_name}.txt"
        file_path.write_text(qasm_code, encoding="utf-8")

        rows.append({
            "Algoritmo": name.split(". ", 1)[1] if ". " in name else name,
            "Qubits": info["n_qubits"],
            "Columnas": info["n_cols"],
            "Descripcion": data.get("desc", ""),
            "Archivo": str(file_path.name)
        })

    df = pd.DataFrame(rows)
    return df


qasm_table = generate_qasm_table_and_files()
qasm_table


,Algoritmo,Qubits,Columnas,Descripcion,Archivo
0,Shor,4,13,Algoritmo de factorización de Shor (4 qubits),1__Shor.txt
1,Bernstein-Vazirani,4,8,Algoritmo de Bernstein-Vazirani (4 qubits),2__Bernstein-Vazirani.txt
2,Grover,2,8,Algoritmo de búsqueda de Grover (2 qubits),3__Grover.txt
3,Deutsch-Jozsa,4,9,Algoritmo de Deutsch-Jozsa (4 qubits),4__Deutsch-Jozsa.txt
4,Simon,6,8,Algoritmo de Simon (6 qubits),5__Simon.txt
5,TSP,3,6,"Circuito para problema del viajante (TSP, 3 qu...",6__TSP.txt
6,Teleportation,5,8,Protocolo de teleportación cuántica (5 qubits),7__Teleportation.txt
7,Phase Estimation,4,17,Estimación de fase cuántica (4 qubits + compue...,8__Phase_Estimation.txt
8,QFT,3,8,Transformada de Fourier Cuántica (3 qubits),9__QFT.txt
9,QAOA,2,12,Quantum Approximate Optimization Algorithm (2 ...,10__QAOA.txt


In [23]:
import sys
import subprocess
import importlib.util

print('Python:', sys.executable)
if importlib.util.find_spec('pylatexenc') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pylatexenc'])

import pylatexenc
print('pylatexenc:', pylatexenc.__file__)


Python: c:\Users\PC\desarrollo\QTransLIFIA\.venv\Scripts\python.exe
pylatexenc: c:\Users\PC\desarrollo\QTransLIFIA\.venv\Lib\site-packages\pylatexenc\__init__.py


### Generación de imágenes de circuitos traducidos OpenQASM3

In [24]:
from pathlib import Path
from qiskit import QuantumCircuit
from qiskit.qasm3 import loads as qasm3_loads


circuits_dir = Path.cwd() / "circuits_qasm"
circuits_dir.mkdir(exist_ok=True)


def load_qasm_circuit(qasm_text: str):
    text = (qasm_text or "").strip()
    if not text:
        raise ValueError("El contenido QASM está vacío")

    upper = text.upper()
    if upper.startswith("OPENQASM 3.0"):
        return qasm3_loads(text)
    if upper.startswith("OPENQASM 2.0"):
        return QuantumCircuit.from_qasm_str(text)

    try:
        return qasm3_loads(text)
    except Exception:
        return QuantumCircuit.from_qasm_str(text)


def render_qasm_circuits():
    qasm_dir = Path.cwd() / "algorithms_qasm"
    circuits_dir = Path.cwd() / "circuits_qasm"
    circuits_dir.mkdir(exist_ok=True)

    if not qasm_dir.exists():
        raise FileNotFoundError(f"No existe la carpeta de QASM: {qasm_dir}")

    files = sorted(qasm_dir.glob("*.txt"))
    if not files:
        raise FileNotFoundError(f"No hay archivos .txt en {qasm_dir}")

    for txt_path in files:
        qasm_text = txt_path.read_text(encoding="utf-8")

        try:
            circuit = load_qasm_circuit(qasm_text)
        except Exception as exc:
            print(f"No se pudo cargar {txt_path.name}: {exc}")
            continue

        png_path = circuits_dir / f"{txt_path.stem}.png"
        try:
            circuit.draw(output='mpl', filename=str(png_path), style='bw')
            print(f"Generado: {png_path.name}")
        except Exception as exc:
            print(f"No se pudo dibujar {txt_path.name}: {exc}")


render_qasm_circuits()


Generado: 10__QAOA.png
Generado: 11__Kickback.png
Generado: 12__Full_Adder.png
Generado: 13__Multicontroled_Gates.png
Generado: 1__Shor.png
Generado: 2__Bernstein-Vazirani.png
Generado: 3__Grover.png
Generado: 4__Deutsch-Jozsa.png
Generado: 5__Simon.png
Generado: 6__TSP.png
Generado: 7__Teleportation.png
Generado: 8__Phase_Estimation.png
Generado: 9__QFT.png


### Generación de imágenes de circuitos originales Quirk

In [25]:
from pathlib import Path

quirk_dir = Path.cwd() / "circuits_quirk"
quirk_dir.mkdir(exist_ok=True)

for name, data in ALGORITHMS.items():
    url = data["url"]
    safe_name = name.replace(".", "_").replace(" ", "_")
    png_path = quirk_dir / f"{safe_name}.png"

    try:
        qasm_code = quirk_to_qasm(url, data.get("offset", 0))
        circuit = load_qasm_circuit(qasm_code)
        circuit.draw(output='mpl', filename=str(png_path), style='bw')
        print(f"Generado Quirk: {png_path.name}")
    except Exception as exc:
        print(f"No se pudo generar Quirk {safe_name}: {exc}")


Generado Quirk: 1__Shor.png
Generado Quirk: 2__Bernstein-Vazirani.png
Generado Quirk: 3__Grover.png
Generado Quirk: 4__Deutsch-Jozsa.png
Generado Quirk: 5__Simon.png
Generado Quirk: 6__TSP.png
Generado Quirk: 7__Teleportation.png
Generado Quirk: 8__Phase_Estimation.png
Generado Quirk: 9__QFT.png
Generado Quirk: 10__QAOA.png
Generado Quirk: 11__Kickback.png
Generado Quirk: 12__Full_Adder.png
Generado Quirk: 13__Multicontroled_Gates.png
